# Set Up


## Libraries

In [2]:
import os
import sys
import re

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    StopWordsRemover,
    CountVectorizer
)
from pyspark.ml.clustering import LDA

## Path

In [3]:
# S3 locations
S3_BUCKET = "s3a://dsan6000-jg2292/ancient-battles"

CLEANED_DIR = f"{S3_BUCKET}/processed/cleaned_articles"

NLP_DIR = f"{S3_BUCKET}/processed/nlp"

LDA_DIR = f"{S3_BUCKET}/models/lda"

# Initialize Spark

In [4]:
# Spark configuration
SPARK_APP_NAME = "Ancient Battles - LDA"

SPARK_MASTER = "spark://ip-172-31-46-25.ec2.internal:7077"

# Initialize Spark
spark = (
    SparkSession.builder
    .appName(SPARK_APP_NAME)
    .master(SPARK_MASTER)
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.5.0"
    )
    .getOrCreate()
)

spark.conf.set(
    "spark.sql.execution.pythonUDF.arrow.enabled",
    "false"
)

# Display Spark configuration
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

:: loading settings :: url = jar:file:/home/ubuntu/spark-4.3.0-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2.5.2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-66a74347-64a4-489d-a497-2b9eaa838230;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 446ms :: artifacts dl 27ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 from central in [default]
	software.amazon.s3.an

Spark version: 4.3.0
Spark master: spark://ip-172-31-46-25.ec2.internal:7077
Application ID: app-20261115223852-0002
Default parallelism: 8


### Verify distributed execution

In [5]:
# Check available parallelism
print("Default parallelism:", spark.sparkContext.defaultParallelism)

# Test distributed execution
test_rdd = spark.sparkContext.parallelize(range(1, 101), 8)

print("Test sum:", test_rdd.sum())
print("Number of partitions:", test_rdd.getNumPartitions())

Default parallelism: 8


Test sum: 5050
Number of partitions: 8


Python version: 3.14.4 (main, Apr 14 2026, 14:26:14) [Clang 22.1.3 ]
Python executable: /home/ubuntu/dsan6000/DSAN600-AncientBattles/.venv/bin/python
PYSPARK_PYTHON: None
PYSPARK_DRIVER_PYTHON: None
